### **Tratamento dos microdados de Eficiência Acadêmica**

In [1]:
import pandas as pd

# Seleção das colunas de interesse para o DataFrame final
colunas_interesse = [
    'Código do Ciclo Matricula',
    'Instituição',               
    'Unidade de Ensino',         
    'Nome de Curso',             
    'Tipo de Curso',
    'Fator Esforço Curso',       
    'Carga Horaria',                              
]

Na análise inicial da estrutura do dataframe, constatamos que a coluna `'tipo_oferta'` apresenta apenas valores registrados como `['Não se aplica']`, isto é, não temos registros do tipo de ingresso para esse conjunto de dados. Por esse motivo, a coluna em questão foi removida.

**Carga dos dados**

In [2]:
# Carregar o arquivo bruto em blocos
chunks = pd.read_csv('../data/raw/microdados_eficiencia_academica_2023.csv', sep=';', encoding='utf-8', usecols=colunas_interesse, chunksize=50000)

# Definir os tipos de graduação para filtrar os dados
tipos_graduacao = ['Licenciatura', 'Bacharelado', 'Tecnologia']

# Aplicar os filtros e concatenar os resultados em um DataFrame final
df_saida = pd.concat(
    [chunk[
        chunk['Instituição'].str.contains('BAHIA', na=False, case=False) &
        chunk['Tipo de Curso'].isin(tipos_graduacao)
    ] for chunk in chunks],

    ignore_index=True
)

# Remover coluna 'Instituição'
df_saida = df_saida.drop(columns=['Instituição'])

In [3]:
# Visualizar tabela resultante
df_saida.info()

<class 'pandas.DataFrame'>
RangeIndex: 2241 entries, 0 to 2240
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Carga Horaria              2241 non-null   int64
 1   Código do Ciclo Matricula  2241 non-null   int64
 2   Fator Esforço Curso        2241 non-null   str  
 3   Nome de Curso              2241 non-null   str  
 4   Tipo de Curso              2241 non-null   str  
 5   Unidade de Ensino          2241 non-null   str  
dtypes: int64(2), str(4)
memory usage: 218.1 KB


In [4]:
df_saida.head()

,Carga Horaria,Código do Ciclo Matricula,Fator Esforço Curso,Nome de Curso,Tipo de Curso,Unidade de Ensino
0,3630,2146517,"1,108",Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
1,3630,2146517,"1,108",Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
2,3630,2146517,"1,108",Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
3,3630,2146517,"1,108",Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
4,3630,2146517,"1,108",Engenharia Elétrica,Bacharelado,Campus Paulo Afonso


Aqui constatamos que essa tabela contém registros múltiplos para cada coluna, já que ela se trata de dados sobre ciclos/turmas e não matrículas individualizadas de estudantes. Por esse motivo, aplicamos o processo de deduplicação nessa base de dados na etapa seguinte.

In [5]:
# Contar repetições por ciclo (= quantidade de alunos por turma no arquivo bruto)
alunos_por_ciclo = df_saida['Código do Ciclo Matricula'].value_counts().reset_index()
alunos_por_ciclo.columns = ['Código do Ciclo Matricula', 'total_alunos']

print(f"Total de ciclos únicos: {len(alunos_por_ciclo)}")
print(f"Média de alunos por ciclo: {alunos_por_ciclo['total_alunos'].mean():.1f}")
print(f"\nDistribuição:")
display(alunos_por_ciclo.sort_values('total_alunos', ascending=False).head(10))
display(alunos_por_ciclo.sort_values('total_alunos', ascending=False).tail(10))
alunos_por_ciclo.describe()


Total de ciclos únicos: 60
Média de alunos por ciclo: 37.4

Distribuição:


,Código do Ciclo Matricula,total_alunos
0,2187094,260
1,2608625,53
2,2500969,51
3,2520993,51
4,2457273,50
5,2499829,50
6,2500966,50
7,2499821,48
8,2500970,48
9,2520997,48


,Código do Ciclo Matricula,total_alunos
50,2569423,25
51,2533057,24
52,2606522,22
53,2512920,20
54,2629676,19
55,2506665,18
56,2553906,17
57,2683707,17
58,2528154,16
59,2642476,1


,Código do Ciclo Matricula,total_alunos
count,6.000000e+01,60.000000
mean,2.537326e+06,37.350000
std,1.222747e+05,31.115327
min,2.146517e+06,1.000000
25%,2.500968e+06,27.000000
50%,2.539134e+06,32.500000
75%,2.608627e+06,41.250000
max,2.742257e+06,260.000000


**Tratamento dos dados**

In [6]:
# Renomear colunas
df_saida.columns = [
    'carga_horaria', 'codigo_ciclo_matricula',
    'fator_esforco_curso', 'nome_curso',
    'tipo_curso', 'unidade_ensino'
]

In [7]:
# Converter o formato da coluna 'fator_esforco_curso' de string para float
df_saida['fator_esforco_curso'] = (
    df_saida['fator_esforco_curso']
    .str.replace(',', '.', regex=False)
    .astype('float64')
)

In [8]:
# Converter o formato da coluna 'codigo_ciclo_matricula' de inteiro para string
df_saida['codigo_ciclo_matricula'] = df_saida['codigo_ciclo_matricula'].astype(str)

In [9]:
# Deduplicar o DataFrame
df_saida = df_saida.drop_duplicates(subset=['codigo_ciclo_matricula']).reset_index(drop=True)

**Persistência do DataFrame final tratado**

In [10]:
# Salvar o DataFrame resultante em arquivo Parquet

df_saida.to_parquet('../data/processed/microdados_eficiencia_academica_tratados.parquet', index=False)